# Chapitre 5 - Exercices 5.16 à 5.19

Régression ridge et noyaux, fonctions propres d'un noyau estimées à partir de la matrice
de Gram, moments nuls des ondelettes, transformée de Haar rapide.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermval
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl.ch05_splines import haar_transform, inverse_haar_transform, haar_matrix
set_style()

## Exercice 5.16 : Ridge et noyau

**Énoncé.** On considère la régression ridge (5.53) avec $M \ge N$ fonctions de base et
un noyau $K(x, y) = \sum_{m=1}^{M}h_m(x)h_m(y) = h(x)^Th(y)$.

(a) Établir (5.62) : $h(x) = \mathbf V\mathbf D_\gamma^{1/2}\phi(x)$. Comment calculer
    $\mathbf V$ et $\mathbf D_\gamma$ à partir de $K$ ? En déduire que le problème
    (5.63), $\min_\beta\sum_i(y_i - \sum_m\beta_mh_m(x_i))^2 + \lambda\sum_m\beta_m^2$,
    équivaut à (5.53), $\min_c\sum_i(y_i - \sum_jc_j\phi_j(x_i))^2 + \lambda\sum_jc_j^2/\gamma_j$.
(b) Montrer (5.75) : $\hat{\mathbf f} = \mathbf H\hat\beta = \mathbf K(\mathbf K +
    \lambda\mathbf I)^{-1}\mathbf y$, où $\mathbf H = (h_m(x_i))$ est $N \times M$ et
    $\mathbf K = \mathbf H\mathbf H^T$.
(c) Montrer (5.76) : $\hat f(x) = h(x)^T\hat\beta = \sum_iK(x, x_i)\hat\alpha_i$ avec
    $\hat\alpha = (\mathbf K + \lambda\mathbf I)^{-1}\mathbf y$.
(d) Que faire si $M < N$ ?

**(a) Lien entre $h$ et la décomposition spectrale.** Les fonctions propres $\phi_i$ de
(5.45) sont orthonormées pour un produit scalaire $\langle u, v\rangle = \int u(x)v(x)\,
d\mu(x)$ (une mesure $\mu$ sur l'espace des $x$, par exemple la loi des données). Posons
$\mathbf G = \int h(x)h(x)^T\,d\mu(x)$, matrice $M \times M$ symétrique semi-définie
positive, et diagonalisons-la : $\mathbf G = \mathbf V\mathbf D_\gamma\mathbf V^T$ avec
$\mathbf V$ orthogonale. Supposons $\gamma_j > 0$ et définissons
$\phi(x) = \mathbf D_\gamma^{-1/2}\mathbf V^Th(x)$, c'est-à-dire
$h(x) = \mathbf V\mathbf D_\gamma^{1/2}\phi(x)$ (5.62). Alors

- $\int\phi\phi^T\,d\mu = \mathbf D_\gamma^{-1/2}\mathbf V^T\mathbf G\mathbf V\mathbf
  D_\gamma^{-1/2} = \mathbf I$ : les $\phi_j$ sont orthonormées ;
- $\sum_j\gamma_j\phi_j(x)\phi_j(y) = \phi(x)^T\mathbf D_\gamma\phi(y) = h(x)^T\mathbf
  V\mathbf D_\gamma^{-1/2}\mathbf D_\gamma\mathbf D_\gamma^{-1/2}\mathbf V^Th(y) =
  h(x)^Th(y) = K(x, y)$.

C'est donc bien la décomposition (5.45) de $K$.

**Calcul à partir de $K$.** On ne connaît que $K$ : on prend pour $\mu$ la mesure
empirique des $x_i$. Alors $\mathbf G = \frac1N\mathbf H^T\mathbf H$, dont les valeurs
propres non nulles sont celles de $\frac1N\mathbf H\mathbf H^T = \frac1N\mathbf K$ (si
$\mathbf H^T\mathbf H v = \gamma v$, alors $\mathbf H\mathbf H^T(\mathbf Hv) = \gamma\,\mathbf Hv$).
Si $\frac1N\mathbf K = \mathbf U\mathbf D_\gamma\mathbf U^T$, les vecteurs propres
correspondants de $\mathbf G$ sont $\mathbf V = \frac{1}{\sqrt N}\mathbf H^T\mathbf
U\mathbf D_\gamma^{-1/2}$ ; quant aux valeurs $\phi(x_i)$, ce sont les lignes de
$\sqrt N\,\mathbf U$ (voir aussi l'exercice 5.17).

**Équivalence de (5.63) et (5.53).** Avec $c = \mathbf D_\gamma^{1/2}\mathbf V^T\beta$
(bijection, car $\mathbf V$ orthogonale et $\gamma_j > 0$) :
$\sum_m\beta_mh_m(x) = \beta^T\mathbf V\mathbf D_\gamma^{1/2}\phi(x) = c^T\phi(x)$ et
$\|\beta\|^2 = \|\mathbf V\mathbf D_\gamma^{-1/2}c\|^2 = c^T\mathbf D_\gamma^{-1}c =
\sum_jc_j^2/\gamma_j$. Les deux critères sont égaux.

**(b)** La solution ridge est $\hat\beta = (\mathbf H^T\mathbf H + \lambda\mathbf
I_M)^{-1}\mathbf H^T\mathbf y$. L'identité « de passage »

$$
(\mathbf H^T\mathbf H + \lambda\mathbf I_M)^{-1}\mathbf H^T = \mathbf H^T(\mathbf
H\mathbf H^T + \lambda\mathbf I_N)^{-1}
$$

se vérifie en partant de $\mathbf H^T(\mathbf H\mathbf H^T + \lambda\mathbf I) =
(\mathbf H^T\mathbf H + \lambda\mathbf I)\mathbf H^T$ et en multipliant à gauche par
$(\mathbf H^T\mathbf H + \lambda\mathbf I)^{-1}$ et à droite par $(\mathbf H\mathbf H^T +
\lambda\mathbf I)^{-1}$ (inversibles pour $\lambda > 0$, car symétriques définies
positives). Donc $\hat{\mathbf f} = \mathbf H\hat\beta = \mathbf H\mathbf H^T(\mathbf
H\mathbf H^T + \lambda\mathbf I)^{-1}\mathbf y = \mathbf K(\mathbf K + \lambda\mathbf
I)^{-1}\mathbf y$.

**(c)** De même $\hat f(x) = h(x)^T\hat\beta = h(x)^T\mathbf H^T(\mathbf K +
\lambda\mathbf I)^{-1}\mathbf y = \sum_ih(x)^Th(x_i)\,\hat\alpha_i = \sum_iK(x,
x_i)\hat\alpha_i$, avec $\hat\alpha = (\mathbf K + \lambda\mathbf I)^{-1}\mathbf y$.

**(d)** Si $M < N$, $\mathbf K = \mathbf H\mathbf H^T$ ($N \times N$) est de rang au plus
$M$, donc singulière ; mais $\mathbf K + \lambda\mathbf I$ reste inversible pour
$\lambda > 0$ et les formules restent **valables**. Elles sont simplement moins
économiques : résoudre le système $M \times M$ du problème primal,
$(\mathbf H^T\mathbf H + \lambda\mathbf I)\beta = \mathbf H^T\mathbf y$, coûte
$O(NM^2 + M^3)$ contre $O(N^3)$ pour la forme à noyau. (Dans la décomposition (a), seules
$M$ valeurs propres $\gamma_j$ sont non nulles.) On utilise donc la forme primale quand
$M < N$ et la forme duale (noyau) quand $M \ge N$.

**Vérification numérique** avec le noyau polynomial $K(x, y) = (\langle x, y\rangle +
1)^2$ sur $\mathbb R^2$ ($M = 6$, base (5.61)), d'abord avec $N = 4 < M$ puis
$N = 200 > M$.

In [2]:
def h_poly(X):
    """Caractéristiques (5.61) du noyau (<x, y> + 1)^2 sur R^2."""
    x1, x2 = X[:, 0], X[:, 1]
    return np.column_stack([np.ones(len(X)), np.sqrt(2) * x1, np.sqrt(2) * x2,
                            x1 ** 2, x2 ** 2, np.sqrt(2) * x1 * x2])

rng = np.random.RandomState(0)
lam = 0.5
for N in [4, 200]:
    X = rng.randn(N, 2)
    y = X[:, 0] ** 2 - X[:, 0] * X[:, 1] + 0.3 * rng.randn(N)
    H = h_poly(X)
    K = (X @ X.T + 1) ** 2
    beta = np.linalg.solve(H.T @ H + lam * np.eye(6), H.T @ y)
    alpha = np.linalg.solve(K + lam * np.eye(N), y)
    X0 = rng.randn(5, 2)
    print("N = %3d : max|HH^T - K| = %.1e | max|f primal - f noyau| (apprentissage) = %.1e "
          "| aux nouveaux points = %.1e" % (N, np.abs(H @ H.T - K).max(),
          np.abs(H @ beta - K @ alpha).max(),
          np.abs(h_poly(X0) @ beta - ((X0 @ X.T + 1) ** 2) @ alpha).max()))

N =   4 : max|HH^T - K| = 7.1e-15 | max|f primal - f noyau| (apprentissage) = 8.9e-16 | aux nouveaux points = 2.3e-14
N = 200 : max|HH^T - K| = 1.4e-14 | max|f primal - f noyau| (apprentissage) = 4.4e-14 | aux nouveaux points = 1.4e-14


Vérifions aussi (a) : à partir de la seule matrice $\mathbf K$ (cas $N = 200$), on
construit $\gamma$ et $\phi(x_i)$, puis on contrôle l'orthonormalité empirique des
$\phi_j$ et la relation $h(x_i) = \mathbf V\mathbf D_\gamma^{1/2}\phi(x_i)$.

In [3]:
d, U = np.linalg.eigh(K / N)
d, U = d[::-1][:6], U[:, ::-1][:, :6]           # les M = 6 valeurs propres non nulles
Phi = np.sqrt(N) * U                             # phi_j(x_i)
V = H.T @ U / np.sqrt(N) / np.sqrt(d)            # V = H^T U D^{-1/2} / sqrt(N)
print("gamma =", np.round(d, 4))
print("V orthogonale :", np.allclose(V.T @ V, np.eye(6)))
print("(1/N) Phi^T Phi = I :", np.allclose(Phi.T @ Phi / N, np.eye(6)))
print("h(x_i) = V D^1/2 phi(x_i) :", np.allclose(H, Phi @ np.diag(np.sqrt(d)) @ V.T))

gamma = [4.1137 2.3062 1.9351 1.6448 1.3341 0.3866]
V orthogonale : True
(1/N) Phi^T Phi = I : True
h(x_i) = V D^1/2 phi(x_i) : True


**Coût** : forme primale ($M \times M$) contre forme noyau ($N \times N$) pour $M = 6$
fixé et $N$ croissant.

In [4]:
for N in [500, 1000, 2000]:
    X = rng.randn(N, 2)
    y = rng.randn(N)
    t0 = time.perf_counter()
    H = h_poly(X)
    np.linalg.solve(H.T @ H + lam * np.eye(6), H.T @ y)
    tp = time.perf_counter() - t0
    t0 = time.perf_counter()
    np.linalg.solve((X @ X.T + 1) ** 2 + lam * np.eye(N), y)
    td = time.perf_counter() - t0
    print("N = %4d : primal %.5f s | noyau %.4f s" % (N, tp, td))

N =  500 : primal 0.00006 s | noyau 0.0027 s
N = 1000 : primal 0.00004 s | noyau 0.0147 s
N = 2000 : primal 0.00012 s | noyau 0.0793 s


> **À retenir.** Ridge dans l'espace des caractéristiques ($\hat\beta$, système
> $M \times M$) et ridge à noyau ($\hat\alpha = (\mathbf K + \lambda\mathbf I)^{-1}\mathbf
> y$, système $N \times N$) donnent les **mêmes** prédictions ; on choisit la forme la
> moins coûteuse selon que $M < N$ ou $M \ge N$.

## Exercice 5.17 : Des vecteurs propres de $\mathbf K$ aux fonctions propres de $K$

**Énoncé.** Montrer comment convertir la décomposition spectrale discrète de la matrice
$\mathbf K$ (§5.8.2) en estimations des fonctions propres du noyau $K$.

**Équation aux fonctions propres.** Si les $x_i$ sont i.i.d. de densité $p$, les
fonctions propres de $K$ pour le produit scalaire $\int uv\,p$ vérifient

$$
\int K(x, y)\,\phi_\ell(y)\,p(y)\,dy = \gamma_\ell\,\phi_\ell(x),\qquad \int\phi_\ell^2\,p = 1 .
$$

On remplace l'intégrale par une moyenne empirique (loi des grands nombres) :
$\frac1N\sum_{j=1}^{N}K(x, x_j)\phi_\ell(x_j) \approx \gamma_\ell\phi_\ell(x)$. Aux points
$x = x_i$, c'est un problème de valeurs propres matriciel :
$\frac1N\mathbf K\,u = \gamma\,u$ avec $u = (\phi_\ell(x_1), \dots, \phi_\ell(x_N))$.

**Conversion.** Si $\mathbf K u_\ell = d_\ell u_\ell$ avec $\|u_\ell\| = 1$ :

- valeurs propres : $\hat\gamma_\ell = d_\ell/N$ ;
- valeurs aux points d'apprentissage : $\hat\phi_\ell(x_i) = \sqrt N\,u_{i\ell}$ (la
  normalisation $\frac1N\sum_i\hat\phi_\ell(x_i)^2 = 1$ imite $\int\phi_\ell^2p = 1$) ;
- en un point $x$ quelconque (**extension de Nyström**), on réutilise l'équation
  intégrale :

$$
\hat\phi_\ell(x) = \frac{1}{N\hat\gamma_\ell}\sum_{j=1}^{N}K(x, x_j)\,\hat\phi_\ell(x_j)
  = \frac{\sqrt N}{d_\ell}\sum_{j=1}^{N}K(x, x_j)\,u_{j\ell}.
$$

En $x = x_i$, on retrouve bien $\sqrt N\,u_{i\ell}$ (car $\mathbf Ku_\ell = d_\ell u_\ell$) :
l'extension interpole les valeurs discrètes, et c'est une combinaison lisse des
$K(\cdot, x_j)$.

**Vérification.** Pour le noyau gaussien $K(x, y) = e^{-b(x-y)^2}$ et $p = \mathcal N(0,
\sigma^2)$, les fonctions propres sont connues explicitement : avec $a = 1/(4\sigma^2)$,
$c = \sqrt{a^2 + 2ab}$, $A = a + b + c$, $B = b/A$, on a
$\gamma_k = \sqrt{2a/A}\,B^k$ et $\phi_k(x) \propto e^{-(c-a)x^2}H_k(\sqrt{2c}\,x)$
($H_k$ : polynômes d'Hermite). Comparons-les aux estimations de Nyström.

In [5]:
rng = np.random.RandomState(0)
N, sigma, b = 400, 1.0, 1.0
xs = rng.randn(N) * sigma
Kmat = np.exp(-b * (xs[:, None] - xs[None, :]) ** 2)
dK, UK = np.linalg.eigh(Kmat)
dK, UK = dK[::-1], UK[:, ::-1]
a = 1 / (4 * sigma ** 2)
c = np.sqrt(a ** 2 + 2 * a * b)
A = a + b + c
gam_th = np.sqrt(2 * a / A) * (b / A) ** np.arange(8)
print("gamma estimées  :", np.round(dK[:8] / N, 5))
print("gamma théoriques:", np.round(gam_th, 5))

gamma estimées  : [0.49788 0.25359 0.12447 0.06667 0.03137 0.01446 0.00695 0.003  ]
gamma théoriques: [0.5     0.25    0.125   0.0625  0.03125 0.01562 0.00781 0.00391]
